In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# Step 1: Reinstall SELURUH torch ecosystem bersama-sama supaya CUDA-nya match
!pip install -q torch torchvision torchaudio --force-reinstall

# Step 2: Install sisanya
!pip install -q transformers peft datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 24.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 8.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 4.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 4.4 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 17.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 64.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━

In [2]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
!mkdir -p src/utils data/raw data/processed models/lora_adapter models/group_sae models/selfie_adapter

In [4]:
!cp /kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/data/raw/ESConv.json /kaggle/working/data/raw
!cp /kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/data/raw/heliosbrahma_mental_health_chatbot_dataset.json /kaggle/working/data/raw

In [5]:
%%writefile src/phase1_train_lora.py
import os
import json
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

# ==========================================
# 1. MAIN CONFIGURATION & HYPERPARAMETERS
# ==========================================
MODEL_NAME = "EleutherAI/pythia-160m"
SAMPLE_SIZE = 500 # Limited samples per dataset for fast Proof of Concept

# LoRA Configuration
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LEARNING_RATE = 1e-5
EPOCHS = 3
MAX_SEQ_LENGTH = 1024

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

# ==========================================
# 2. DATASET LOADING & PROCESSING
# ==========================================
def prepare_datasets():
    print("\n📦 Loading and processing datasets...")
    training_texts = []

    # Automatic execution environment detection (Kaggle vs. Local)
    if os.path.exists("/kaggle/input"):
        # ATTENTION: Change 'ortho-selfie-raw-data' to your Kaggle dataset folder name if different
        RAW_DATA_DIR = "/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/data/raw"
        OUTPUT_DIR = "/kaggle/working/models/lora_adapter"
        print(f"🌍 Running in Kaggle mode. Using data path: {RAW_DATA_DIR}")
    else:
        # Strict alignment with our established repo_structure
        BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
        RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
        OUTPUT_DIR = os.path.join(BASE_DIR, "models", "lora_adapter")
        print(f"💻 Running in Local mode. Using data path: {RAW_DATA_DIR}")

    # A. Empathy Dataset (ESConv)
    esconv_path = os.path.join(RAW_DATA_DIR, "ESConv.json")
    try:
        if os.path.exists(esconv_path):
            with open(esconv_path, "r", encoding="utf-8") as f:
                esconv_data = json.load(f)
                count = 0
                for item in esconv_data:
                    if count >= SAMPLE_SIZE: break
                    if "dialog" in item:
                        for turn in item["dialog"]:
                            if turn.get("speaker") == "supporter":
                                text = turn.get("content", "").strip()
                                if text:
                                    training_texts.append(f"Therapist: {text}")
                                    count += 1
                                    if count >= SAMPLE_SIZE: break
            print(f"✅ ESConv (Empathy): {count} samples loaded.")
        else:
            print(f"⚠️ File ESConv.json not found at {esconv_path}, skipping.")
    except Exception as e:
        print(f"❌ Failed to load ESConv: {e}")

    # B. Clinical Dataset (Heliosbrahma)
    helios_path = os.path.join(RAW_DATA_DIR, "heliosbrahma_dataset.json")
    try:
        if os.path.exists(helios_path):
            with open(helios_path, "r", encoding="utf-8") as f:
                count = 0
                for line in f:
                    if count >= SAMPLE_SIZE: break
                    line = line.strip()
                    if not line: continue
                    data = json.loads(line)
                    text_block = data.get("text", "")
                    if "<ASSISTANT>:" in text_block:
                        ast = text_block.split("<ASSISTANT>:")[1].strip()
                        training_texts.append(f"Clinical Diagnosis: {ast}")
                        count += 1
            print(f"✅ Heliosbrahma (Clinical): {count} samples loaded.")
        else:
             print(f"⚠️ File heliosbrahma_dataset.json not found at {helios_path}, skipping.")
    except Exception as e:
         print(f"❌ Failed to load Heliosbrahma: {e}")

    # C. Clinical Classification Dataset (Hugging Face)
    try:
        print("Downloading Clinical Classification dataset from Hugging Face...")
        hf_dataset = load_dataset("sai1908/Mental_Health_Condition_Classification", split="train")
        hf_texts = hf_dataset['text'][:SAMPLE_SIZE]
        hf_labels = hf_dataset['status'][:SAMPLE_SIZE]

        for i in range(len(hf_texts)):
            formatted_text = f"Patient Complaint: {hf_texts[i]}\nAnalysis: The patient exhibits symptoms of {hf_labels[i]}."
            training_texts.append(formatted_text)

        print(f"✅ Mental Health Classification (Clinical): {len(hf_texts)} samples loaded.")
    except Exception as e:
        print(f"❌ Failed to load HF classification dataset: {e}")

    return Dataset.from_dict({"text": training_texts}), OUTPUT_DIR

# ==========================================
# 3. MODEL & TOKENIZER INITIALIZATION
# ==========================================
print(f"\n🧠 Loading model {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto" 
)

# ==========================================
# 4. LoRA CONFIGURATION
# ==========================================
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["query_key_value"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# ==========================================
# 5. TRAINING (FINE-TUNING)
# ==========================================
train_dataset, output_dir = prepare_datasets()
print(f"\nTotal combined training data: {len(train_dataset)} rows.")

# Tokenize dataset explicitly (replaces SFTTrainer's automatic tokenization)
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

print("🔤 Tokenizing dataset...")
tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_torch",
    fp16=True if torch.cuda.is_available() else False, 
    report_to="none",
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("\n🚀 Starting LoRA Fine-Tuning process...")
trainer.train()

# ==========================================
# 6. SAVING TRAINED ADAPTER
# ==========================================
print("\n💾 Saving adapter model...")
os.makedirs(output_dir, exist_ok=True)
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ Phase 1 Completed! LoRA model saved in directory: {os.path.abspath(output_dir)}")

Writing src/phase1_train_lora.py


In [6]:
!python src/phase1_train_lora.py

🖥️ Using device: cuda

🧠 Loading model EleutherAI/pythia-160m...
tokenizer_config.json: 100%|███████████████████| 396/396 [00:00<00:00, 1.46MB/s]
tokenizer.json: 2.11MB [00:00, 47.5MB/s]
special_tokens_map.json: 100%|████████████████| 99.0/99.0 [00:00<00:00, 246kB/s]
model.safetensors: 100%|██████████████████████| 375M/375M [00:03<00:00, 117MB/s]
Loading weights: 100%|█| 148/148 [00:00<00:00, 961.99it/s, Materializing param=g
trainable params: 294,912 || all params: 162,617,856 || trainable%: 0.1814

📦 Loading and processing datasets...
🌍 Running in Kaggle mode. Using data path: /kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/data/raw
✅ ESConv (Empathy): 500 samples loaded.
⚠️ File heliosbrahma_dataset.json not found at /kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/data/raw/heliosbrahma_dataset.json, skipping.
README.md: 100%|███████████████████████████████| 791/791 [00:00<00:00, 2.41MB/s]
Mental Health Text Dataset for Emotion a(…): 100%|█| 46.5M/

In [14]:
%%writefile src/phase2_train_sae.py
import os
import time
import json
import itertools
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

# ==========================================
# 1. MAIN CONFIGURATION & HYPERPARAMETERS
# ==========================================
MODEL_NAME = "EleutherAI/pythia-160m"

# Smart Path Resolver (Adapts to Kaggle session, Kaggle input, or Local execution)
if os.path.exists("/kaggle/working/models/lora_adapter/adapter_config.json"):
    # Case A: Running in the same Kaggle session right after Phase 1
    RAW_DATA_DIR = "/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/data/raw" 
    LORA_DIR = "/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/models/lora_adapter"
    OUTPUT_DIR = "/kaggle/working/models/group_sae"
    print(f"🌍 Running in Kaggle (Same Session). Output path: {OUTPUT_DIR}")
elif os.path.exists("/kaggle/input"):
    # Case B: Running in a new Kaggle session with datasets attached
    # ATTENTION: Adjust 'ortho-selfie-lora-pythia' to your actual dataset name if uploaded
    RAW_DATA_DIR = "/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/data/raw" 
    LORA_DIR = "/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/models/lora_adapter" 
    OUTPUT_DIR = "/kaggle/working/models/group_sae"
    print(f"🌍 Running in Kaggle (Attached Dataset). Output path: {OUTPUT_DIR}")
else:
    # Case C: Strict alignment with local repo_structure
    BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
    LORA_DIR = os.path.join(BASE_DIR, "models", "lora_adapter")
    OUTPUT_DIR = os.path.join(BASE_DIR, "models", "group_sae")
    print(f"💻 Running in Local mode. Output path: {OUTPUT_DIR}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# SAE Hyperparameters (Aligned with Group-SAE literature)
EXPANSION_FACTOR = 16 
TOP_K = 128
INPUT_DIM = 768  # Pythia-160M hidden dimension
HIDDEN_DIM = INPUT_DIM * EXPANSION_FACTOR
BATCH_SIZE = 2048
LEARNING_RATE = 1e-4

# Orthogonal Penalty (Separating Empathy vs. Clinical Reasoning)
ORTHO_LAMBDA = 10.0  

# Streaming Buffer Capacity (Safe for Kaggle RAM limitations)
BUFFER_TEXT_LIMIT = 750 
EPOCHS = 6
SAMPLE_SIZE = 750 # Limited texts per dataset for PoC

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

# ==========================================
# 2. GROUP-SAE ARCHITECTURE
# ==========================================
class TopK_SAE(nn.Module):
    """Sparse Autoencoder using Top-K activation."""
    def __init__(self, input_dim, hidden_dim, k):
        super().__init__()
        self.k = k
        self.encoder = nn.Linear(input_dim, hidden_dim, bias=True)
        self.decoder = nn.Linear(hidden_dim, input_dim, bias=True)
        
        nn.init.kaiming_uniform_(self.encoder.weight)
        nn.init.zeros_(self.encoder.bias)
        nn.init.kaiming_uniform_(self.decoder.weight)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, x):
        encoded = self.encoder(x)
        topk_values, topk_indices = torch.topk(encoded, self.k, dim=-1)
        sparse_encoded = torch.zeros_like(encoded).scatter_(-1, topk_indices, topk_values)
        sparse_encoded = torch.relu(sparse_encoded)
        reconstructed = self.decoder(sparse_encoded)
        return reconstructed, sparse_encoded

def calculate_fvu(original, reconstructed):
    """Calculating Fraction of Variance Unexplained (FVU)."""
    mse = torch.nn.functional.mse_loss(reconstructed, original, reduction='mean')
    variance = torch.var(original, unbiased=False)
    fvu = mse / (variance + 1e-8) 
    return fvu.item()

# ==========================================
# 3. DATASET LOADING (EMPATHY VS CLINICAL)
# ==========================================
def load_datasets():
    print("\n=== Loading Datasets for Two Manifolds ===")
    
    empathy_texts = []
    clinical_texts = []

    # A. EMPATHY MANIFOLD (ESConv)
    esconv_path = os.path.join(RAW_DATA_DIR, "ESConv.json")
    try:
        if os.path.exists(esconv_path):
            with open(esconv_path, "r", encoding="utf-8") as f:
                esconv_data = json.load(f)
                count = 0
                for item in esconv_data:
                    if count >= SAMPLE_SIZE: break
                    if "dialog" in item:
                        for turn in item["dialog"]:
                            if turn.get("speaker") == "supporter":
                                text = turn.get("content", "").strip()
                                if text:
                                    empathy_texts.append(text)
                                    count += 1
                                    if count >= SAMPLE_SIZE: break
            print(f"✅ Empathy Manifold (ESConv): {len(empathy_texts)} samples.")
        else:
            print(f"⚠️ File ESConv.json not found, skipping.")
    except Exception as e:
        print(f"❌ Failed to load ESConv: {e}")

    # B. CLINICAL MANIFOLD (Heliosbrahma)
    helios_path = os.path.join(RAW_DATA_DIR, "heliosbrahma_dataset.json")
    try:
        if os.path.exists(helios_path):
            with open(helios_path, "r", encoding="utf-8") as f:
                count = 0
                for line in f:
                    if count >= SAMPLE_SIZE: break
                    line = line.strip()
                    if not line: continue
                    data = json.loads(line)
                    text_block = data.get("text", "")
                    if "<ASSISTANT>:" in text_block:
                        ast = text_block.split("<ASSISTANT>:")[1].strip()
                        clinical_texts.append(ast)
                        count += 1
            print(f"✅ Clinical Manifold (Heliosbrahma): {count} samples.")
        else:
            print(f"⚠️ File heliosbrahma_dataset.json not found, skipping.")
    except Exception as e:
        print(f"❌ Failed to load Heliosbrahma: {e}")

    # C. ADDITIONAL CLINICAL MANIFOLD (Hugging Face)
    try:
        print("Downloading Clinical Classification dataset from Hugging Face...")
        hf_dataset = load_dataset("sai1908/Mental_Health_Condition_Classification", split="train")
        hf_texts = hf_dataset['text'][:SAMPLE_SIZE]
        clinical_texts.extend(hf_texts)
        print(f"✅ Clinical Manifold (HF Mental Health): {len(hf_texts)} samples.")
    except Exception as e:
        print(f"❌ Failed to load HF classification dataset: {e}")
        
    return empathy_texts, clinical_texts

# ==========================================
# 4. ACTIVATION EXTRACTION & TRAINING
# ==========================================
def get_token_activations(text, llm_model, tokenizer, device, group_layers):
    """Extract activations from specific layers dynamically."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64).to(device)
    with torch.no_grad():
        outputs = llm_model(**inputs, output_hidden_states=True)
        # Fetch hidden states, skip initial embedding layer (index 0 usually denotes embeddings)
        hidden_states = outputs.hidden_states[1:] 
        
        layer_acts = []
        for layer_idx in group_layers:
            state = hidden_states[layer_idx].squeeze(0).cpu()
            layer_acts.append(state)
            
        return torch.cat(layer_acts, dim=0)

def train_streaming_sae(actual_groups, empathy_texts, clinical_texts, llm_model, tokenizer, device):
    """Dynamic Buffer Training Cycle with Orthogonal Penalty."""
    
    for idx, group_layers in enumerate(actual_groups):
        group_id = idx + 1
        print(f"\n{'='*60}")
        print(f"=== Starting Streaming Group-SAE {group_id} ===")
        print(f"Target Layers: {group_layers}")
        
        sae_model = TopK_SAE(INPUT_DIM, HIDDEN_DIM, TOP_K).to(device)
        optimizer = optim.Adam(sae_model.parameters(), lr=LEARNING_RATE)
        mse_loss_fn = nn.MSELoss()
        sae_model.train()
        
        for epoch in range(EPOCHS):
            print(f"\n  --- Epoch {epoch+1}/{EPOCHS} ---")
            
            clin_iter = itertools.cycle(clinical_texts)
            emp_iter = itertools.cycle(empathy_texts)
            
            max_texts = max(len(clinical_texts), len(empathy_texts))
            text_processed = 0
            total_steps = 0  
            step_start_time = time.time() 
            
            buffer_clin = []
            buffer_emp = []
            
            while text_processed < max_texts:
                # 1. Activation Caching Phase (Filling the Buffer)
                while len(buffer_clin) < BUFFER_TEXT_LIMIT and text_processed < max_texts:
                    t_clin = next(clin_iter)
                    acts_c = get_token_activations(t_clin, llm_model, tokenizer, device, group_layers)
                    buffer_clin.append(acts_c)
                    
                    t_emp = next(emp_iter)
                    acts_e = get_token_activations(t_emp, llm_model, tokenizer, device, group_layers)
                    buffer_emp.append(acts_e)
                    
                    text_processed += 1
                    
                if not buffer_clin or not buffer_emp:
                    break
                    
                # 2. Training Phase (Draining the Buffer)
                print(f"  [Epoch {epoch+1}] Training from buffer... (Text Progress: {text_processed}/{max_texts})")
                tensor_clin = torch.cat(buffer_clin, dim=0)
                tensor_emp = torch.cat(buffer_emp, dim=0)
                
                # Shuffle tokens to prevent overfitting
                tensor_clin = tensor_clin[torch.randperm(tensor_clin.size(0))]
                tensor_emp = tensor_emp[torch.randperm(tensor_emp.size(0))]
                
                num_tokens = min(tensor_clin.size(0), tensor_emp.size(0))
                
                for i in range(0, num_tokens, BATCH_SIZE):
                    x_clin = tensor_clin[i:i+BATCH_SIZE].to(device, dtype=torch.float32)
                    x_emp = tensor_emp[i:i+BATCH_SIZE].to(device, dtype=torch.float32)
                    
                    if x_clin.size(0) < BATCH_SIZE:
                        continue
                        
                    optimizer.zero_grad()
                    
                    recon_clin, sparse_clin = sae_model(x_clin)
                    loss_clin = mse_loss_fn(recon_clin, x_clin)
                    
                    recon_emp, sparse_emp = sae_model(x_emp)
                    loss_emp = mse_loss_fn(recon_emp, x_emp)
                    
                    # Compute Cosine Similarity for Orthogonal Penalty
                    mean_clin = sparse_clin.mean(dim=0)
                    mean_emp = sparse_emp.mean(dim=0)
                    ortho_penalty = F.cosine_similarity(mean_clin.unsqueeze(0), mean_emp.unsqueeze(0), eps=1e-8).squeeze()
                    
                    # TOTAL LOSS = Clinical Recon + Empathy Recon + (Lambda * Overlap Penalty)
                    loss = loss_clin + loss_emp + (ORTHO_LAMBDA * ortho_penalty)
                    loss.backward()
                    optimizer.step()
                    
                    total_steps += 1
                    
                    if total_steps % 50 == 0:
                        elapsed_time = time.time() - step_start_time 
                        fvu_c = calculate_fvu(x_clin, recon_clin)
                        fvu_e = calculate_fvu(x_emp, recon_emp)
                        
                        print(f"    Step {total_steps:04d} | Loss: {loss.item():.4f} | Ortho Pen: {ortho_penalty.item():.4f} | FVU Clin: {fvu_c:.4f} | FVU Emp: {fvu_e:.4f} | Time/50 steps: {elapsed_time:.2f}s")
                        step_start_time = time.time() 
                        
                # 3. Clear RAM Buffer
                buffer_clin = []
                buffer_emp = []
        
        # Save SAE Weights
        save_path = os.path.join(OUTPUT_DIR, f"group_sae_clinical_group_{group_id}.pt")
        torch.save(sae_model.state_dict(), save_path)
        print(f"✅ Group-SAE {group_id} successfully saved to {save_path}")

# ==========================================
# 5. MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    print(f"\n🧠 Loading tokenizer for {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    print(f"🧠 Loading Base Model ({MODEL_NAME})...")
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")
    
    print(f"🔗 Attaching LoRA Adapter from Phase 1 ({LORA_DIR})...")
    try:
        # Load the fine-tuned LoRA model
        llm_model = PeftModel.from_pretrained(base_model, LORA_DIR)
        llm_model.eval() # Ensure model is frozen for activation extraction
        print("✅ Fine-tuned LLM ready for extraction!")
    except Exception as e:
        print(f"❌ Failed to load LoRA Adapter. Did Phase 1 complete successfully? Error: {e}")
        exit()
    
    empathy_texts, clinical_texts = load_datasets()
    
    # Define Layer Groups Architecture
    # Pythia-160M has 12 layers (0 to 11). Grouping early vs middle-late layers.
    if empathy_texts and clinical_texts:
        actual_groups = [
            [0],                                       # Group 1: Layer 0
            [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]        # Group 2: Layers 1-11
        ]
        train_streaming_sae(actual_groups, empathy_texts, clinical_texts, llm_model, tokenizer, device)
    else:
        print("⚠️ Execution aborted: Datasets are empty or failed to load.")

Overwriting src/phase2_train_sae.py


In [15]:
!python src/phase2_train_sae.py

🌍 Running in Kaggle (Same Session). Output path: /kaggle/working/models/group_sae
🖥️ Using device: cuda

🧠 Loading tokenizer for EleutherAI/pythia-160m...
🧠 Loading Base Model (EleutherAI/pythia-160m)...
Loading weights: 100%|█| 148/148 [00:00<00:00, 956.31it/s, Materializing param=g
🔗 Attaching LoRA Adapter from Phase 1 (/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/models/lora_adapter)...
✅ Fine-tuned LLM ready for extraction!

=== Loading Datasets for Two Manifolds ===
✅ Empathy Manifold (ESConv): 750 samples.
⚠️ File heliosbrahma_dataset.json not found, skipping.
✅ Clinical Manifold (HF Mental Health): 750 samples.

=== Starting Streaming Group-SAE 1 ===
Target Layers: [0]

  --- Epoch 1/6 ---
  [Epoch 1] Training from buffer... (Text Progress: 750/750)

  --- Epoch 2/6 ---
  [Epoch 2] Training from buffer... (Text Progress: 750/750)

  --- Epoch 3/6 ---
  [Epoch 3] Training from buffer... (Text Progress: 750/750)

  --- Epoch 4/6 ---
  [Epoch 4] Training from b

In [9]:
%%writefile src/phase3_train_selfie.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ==========================================
# 1. MAIN CONFIGURATION & HYPERPARAMETERS
# ==========================================
MODEL_NAME = "EleutherAI/pythia-160m"

# Smart Path Resolver for Kaggle/Local Execution
if os.path.exists("/kaggle/working/models/lora_adapter/adapter_config.json/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/models/lora_adapter/adapter_config.json"):
    LORA_DIR = "/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/models/lora_adapter"
    SAE_DIR = "/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/models/group_sae"
    OUTPUT_DIR = "/kaggle/working/models/selfie_adapter"
    print(f"🌍 Running in Kaggle (Same Session).")
elif os.path.exists("/kaggle/input"):
    # ATTENTION: Adjust these paths to your actual Kaggle dataset names if running in a new session
    LORA_DIR = "/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/models/lora_adapter"
    SAE_DIR = "/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/models/group_sae"
    OUTPUT_DIR = "/kaggle/working/models/selfie_adapter"
    print(f"🌍 Running in Kaggle (Attached Datasets).")
else:
    BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    LORA_DIR = os.path.join(BASE_DIR, "models", "lora_adapter")
    SAE_DIR = os.path.join(BASE_DIR, "models", "group_sae")
    OUTPUT_DIR = os.path.join(BASE_DIR, "models", "selfie_adapter")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters based on Group-SAE literature
EXPANSION_FACTOR = 16 
TOP_K = 128
INPUT_DIM = 768  
HIDDEN_DIM = INPUT_DIM * EXPANSION_FACTOR

# SelfIE Adapter Hyperparameters
LEARNING_RATE = 1e-3
EPOCHS = 10
BATCH_SIZE = 8
NUM_TRAIN_SAMPLES = 500 # Using a subset of latents for fast PoC execution

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

# ==========================================
# 2. ARCHITECTURES
# ==========================================
class TopK_SAE(nn.Module):
    """Sparse Autoencoder architecture to load Phase 2 weights."""
    def __init__(self, input_dim, hidden_dim, k):
        super().__init__()
        self.k = k
        self.encoder = nn.Linear(input_dim, hidden_dim, bias=True)
        self.decoder = nn.Linear(hidden_dim, input_dim, bias=True)

class ScalarAffineAdapter(nn.Module):
    """
    Lightweight adapter to transform SAE latent vectors into LLM activation space.
    Contains only d_model + 1 parameters (Scale and Bias).
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(1))
        self.bias = nn.Parameter(torch.zeros(hidden_dim))

    def forward(self, h):
        """ Transforms the latent vector h -> f(h) """
        return (self.scale * h) + self.bias

# ==========================================
# 3. LATENT EXTRACTION & AUTO-INTERPRETABILITY
# ==========================================
def extract_sae_vectors(sae_path, device):
    """Extracts the actual geometric feature directions from the SAE decoder."""
    print(f"\n🔍 Extracting real latent vectors from Group-SAE...")
    sae = TopK_SAE(INPUT_DIM, HIDDEN_DIM, TOP_K)
    
    try:
        sae.load_state_dict(torch.load(sae_path, map_location=device))
    except Exception as e:
        print(f"❌ Failed to load SAE weights at {sae_path}. Error: {e}")
        exit()
        
    # The decoder weights map hidden_dim -> input_dim. 
    # Shape of weight is (input_dim, hidden_dim). Transposing gives us (hidden_dim, input_dim),
    # where each row is a 768-dimensional latent vector (h).
    latent_vectors = sae.decoder.weight.detach().T
    print(f"✅ Extracted {latent_vectors.shape[0]} latent vectors of dimension {latent_vectors.shape[1]}.")
    return latent_vectors

def generate_labels_with_explainer(num_samples):
    """
    Simulates the Auto-Interpretability pipeline.
    In production, this queries Groq API with activating texts to get real labels.
    """
    print("\n📝 Running Auto-Interpretability Pipeline (Explainer LLM)...")
    text_labels = []
    api_key = os.environ.get("GROQ_API_KEY")
    
    for i in range(num_samples):
        if api_key:
            # Future Implementation: Call Groq API here using llama-3.3-70b-versatile
            pass
        
        # Fallback for PoC to ensure Cross-Entropy training runs smoothly
        desc = f"Concept representation for clinical or empathetic feature {i}"
        
        # Append closing quote and EOS token (Crucial for SelfIE training format)
        formatted_label = f"{desc}\"<|endoftext|>" 
        text_labels.append(formatted_label)
        
    print(f"✅ Generated {num_samples} explanation labels.")
    return text_labels

# ==========================================
# 4. TRAINING LOOP (FIXED COMPUTATION GRAPH)
# ==========================================
def train_selfie_adapter(model, tokenizer, adapter, latent_vectors, text_labels, device):
    print("\n🚀 Starting Trained SelfIE Adapter Training...")
    
    # Ensure Base LLM is completely frozen to preserve general capabilities
    for param in model.parameters():
        param.requires_grad = False
    model.eval() 
    
    adapter.train()
    optimizer = optim.AdamW(adapter.parameters(), lr=LEARNING_RATE)
    cross_entropy_loss = nn.CrossEntropyLoss()
    
    # SelfIE Prompt Template
    prompt_template = "The following latent feature represents: \""
    
    for epoch in range(EPOCHS):
        total_loss = 0.0
        
        for i in range(0, NUM_TRAIN_SAMPLES, BATCH_SIZE):
            # Slicing the real extracted SAE vectors
            batch_h = latent_vectors[i:i+BATCH_SIZE].to(device)
            batch_labels = text_labels[i:i+BATCH_SIZE]
            
            optimizer.zero_grad()
            batch_loss = 0.0
            
            # ⚠️ CRITICAL FIX: Ambil ukuran terkecil agar sisa batch terakhir tidak error
            current_batch_size = min(len(batch_h), len(batch_labels))
            
            for j in range(current_batch_size):
                h = batch_h[j]
                target_text = batch_labels[j]
                
                # Tokenize prompt and target label
                prompt_ids = tokenizer.encode(prompt_template, return_tensors="pt").to(device)
                target_ids = tokenizer.encode(target_text, return_tensors="pt").to(device)
                full_input_ids = torch.cat([prompt_ids, target_ids], dim=1)
                
                # Get standard embeddings (No grad needed just to fetch the base embeddings)
                with torch.no_grad():
                    base_embeds = model.get_input_embeddings()(full_input_ids)
                
                # Clone the embeddings to maintain a safe computation graph
                inputs_embeds = base_embeds.clone()
                
                # Inject transformed activation f(h) at the placeholder position (' "')
                placeholder_idx = prompt_ids.shape[1] - 1
                f_h = adapter(h)
                inputs_embeds[0, placeholder_idx, :] = f_h
                
                # Forward pass WITHOUT torch.no_grad()
                outputs = model(inputs_embeds=inputs_embeds)
                
                logits = outputs.logits[0]
                
                # Calculate Cross-Entropy Loss on target tokens
                shift_logits = logits[placeholder_idx:-1, :].contiguous()
                shift_labels = target_ids[0].contiguous()
                
                loss = cross_entropy_loss(shift_logits, shift_labels)
                batch_loss += loss
            
            # Jangan lupa bagi loss dengan current_batch_size yang baru
            batch_loss = batch_loss / current_batch_size
            batch_loss.backward()
            optimizer.step()
            
            total_loss += batch_loss.item()
            
        avg_loss = total_loss / (NUM_TRAIN_SAMPLES / BATCH_SIZE)
        print(f"  Epoch {epoch+1}/{EPOCHS} | Average Cross-Entropy Loss: {avg_loss:.4f}")

    print("\n✅ Training Complete!")
    return adapter

# ==========================================
# 5. MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    print(f"\n🧠 Loading tokenizer for {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    print(f"🧠 Loading Base Model ({MODEL_NAME})...")
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    
    print(f"🔗 Attaching LoRA Adapter from Phase 1 ({LORA_DIR})...")
    try:
        fine_tuned_model = PeftModel.from_pretrained(base_model, LORA_DIR).to(device)
        print("✅ Fine-tuned LLM loaded successfully!")
    except Exception as e:
        print(f"❌ Failed to load LoRA Adapter. Error: {e}")
        exit()
        
    # Initialize the lightweight Scalar Affine Adapter
    adapter = ScalarAffineAdapter(hidden_dim=INPUT_DIM).to(device)
    
    # Extract real vectors from Phase 2 Group-SAE (Using Group 2 as an example)
    target_sae_file = os.path.join(SAE_DIR, "group_sae_clinical_group_2.pt")
    latent_vecs = extract_sae_vectors(target_sae_file, device)
    
    # Generate labels using the Explainer pipeline
    text_lbls = generate_labels_with_explainer(NUM_TRAIN_SAMPLES)
    
    # Train the Adapter
    trained_adapter = train_selfie_adapter(
        model=fine_tuned_model,
        tokenizer=tokenizer,
        adapter=adapter,
        latent_vectors=latent_vecs,
        text_labels=text_lbls,
        device=device
    )
    
    # Save the Adapter weights
    save_path = os.path.join(OUTPUT_DIR, "trained_selfie_adapter.pt")
    torch.save(trained_adapter.state_dict(), save_path)
    print(f"💾 Trained SelfIE Adapter successfully saved to {save_path}")

Overwriting src/phase3_train_selfie.py


In [10]:
!python src/phase3_train_selfie.py

🌍 Running in Kaggle (Attached Datasets).
🖥️ Using device: cuda

🧠 Loading tokenizer for EleutherAI/pythia-160m...
🧠 Loading Base Model (EleutherAI/pythia-160m)...
Loading weights: 100%|█| 148/148 [00:00<00:00, 1697.45it/s, Materializing param=
🔗 Attaching LoRA Adapter from Phase 1 (/kaggle/input/datasets/narendrabayutama/ortho-groupsae-selfie-poc/models/lora_adapter)...
✅ Fine-tuned LLM loaded successfully!

🔍 Extracting real latent vectors from Group-SAE...
✅ Extracted 12288 latent vectors of dimension 768.

📝 Running Auto-Interpretability Pipeline (Explainer LLM)...
✅ Generated 500 explanation labels.

🚀 Starting Trained SelfIE Adapter Training...
  Epoch 1/10 | Average Cross-Entropy Loss: 5.0270
  Epoch 2/10 | Average Cross-Entropy Loss: 1.4219
  Epoch 3/10 | Average Cross-Entropy Loss: 0.6736
  Epoch 4/10 | Average Cross-Entropy Loss: 0.6328
  Epoch 5/10 | Average Cross-Entropy Loss: 0.6015
  Epoch 6/10 | Average Cross-Entropy Loss: 0.5868
  Epoch 7/10 | Average Cross-Entropy Loss: